<a href="https://colab.research.google.com/github/roberto-chavez/auto-prompt-sam2/blob/_E.V_/notebooks/07_ablaciones_y_seleccion_ejecutado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 — Ablaciones y selección (solo validación)

Implementación preparada el 2026-11-01 (America/Lima). **Etapa abierta hasta revisión de evidencia Colab de ambos regímenes.** Leer `docs/project/validacion_etapa7.md` y el cuaderno de decisiones antes de ejecutar.

Solo el usuario ejecuta modelos en Colab GPU. Sin entrenamiento, recalibración, test, Bolivia, Paraguay reservado ni demo. Las selecciones y locks se generan con mediciones reales; no existen resultados nuevos por crear este notebook.


## Hipótesis y controles previos

H1 WAIMLAP: referencia asistida > puntos DINOv3 > malla. H2: positivos mejoran hasta saturar; negativos reducen FP. H3: ajustar decoder ayuda más a eventos vistos que a Bolivia; **la parte geográfica no se contrasta aquí**.

A1: cuatro fuentes, decoder base congelado. Referencia asistida emparejada 3+/3− y distancia 16 px; se conserva además la referencia histórica 1+/1− de etapa 3. A2: k+ {1,3,5,10} con k−=3, después k− {0,1,3,5} con k+=3, recomendaciones registradas y sin reajuste de umbrales. A3: misma ruta directa de etapa 6 en ambas ramas, puntos/probe/umbrales idénticos, solo cambia decoder. No reutilizar automáticamente la ruta predictor como control de la ruta directa.

13 filas y 10 contrastes por régimen; 89/75 chips val. Todos los resultados, fallos y deltas negativos se conservan. NDWI aplazado/excluido. La selección automática usa IoU agua pooled, empate exacto por menor latencia desde imagen y luego ID estable. La calibración de etapa 5 y best de etapa 6 se registran aparte, sin nuevas búsquedas.


## 1. Entorno y paquete de fuentes

En PC ejecutar `scripts/validate_stage7_local.py` con el Python de `.venv-validation`; subir `artifacts/auto-prompt-sam2-stage7.zip` a MyDrive. Copiar su SHA en la siguiente celda. Mantener la raíz histórica completa con features y checkpoints de etapas 1–6; los ZIP de evidencia ligera no la reemplazan. Elegir runtime Colab con GPU. La instalación reconstruye el entorno fijado de Colab, no un entorno de modelos en PC.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2'
REPO = Path('/content/auto-prompt-sam2-stage7')
PYTHON = Path('/content/auto-prompt-venv/bin/python')
BUNDLE = Path('/content/drive/MyDrive/auto-prompt-sam2-stage7.zip')
BUNDLE = Path('/content/drive/MyDrive/auto-prompt-sam2-stage7-reparado.zip')
EXPECTED_BUNDLE_SHA256 = 'e9556df886da8bd6585f8c3a29558fdd0c2fea052a2ab96a74e60025dbe94638'
assert len(EXPECTED_BUNDLE_SHA256) == 64, 'Configura el SHA-256 del paquete.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for item in archive.infolist():
        assert (REPO / item.filename).resolve().is_relative_to(REPO.resolve())
    archive.extractall(REPO)
child_env = dict(os.environ, PYTHONPATH=str(REPO / 'src'),
    AUTO_PROMPT_PERSIST_ROOT=PERSIST_ROOT, PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8',
    AUTO_PROMPT_EXECUTION_CONTEXT='google_colab_user_notebook')
# Todos los modelos se recargan offline desde las generaciones autorizadas de Drive.
LOG_DIR = Path(PERSIST_ROOT) / 'stage7/notebook_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
def command(args, log_name):
    import time
    log_path = LOG_DIR / (str(time.time_ns()) + '_' + log_name)
    print('Log persistente:', log_path)
    with log_path.open('x', encoding='utf-8') as log:
        process = subprocess.Popen(args, cwd=REPO, env=child_env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        result = process.wait()
    if result:
        raise RuntimeError(f'Proceso terminado con código {result}; conservar salida y log {log_path}')
if not PYTHON.is_file():
    command([sys.executable, str(REPO / 'scripts/bootstrap_colab.py')], 'bootstrap.log')
command([str(PYTHON), '-m', 'pip', 'install', '--no-deps', '-r', str(REPO / 'requirements-stage2.txt')], 'stage2_pins.log')
command([str(PYTHON), '-m', 'pip', 'check'], 'pip_check.log')


Mounted at /content/drive
Log persistente: /content/drive/MyDrive/auto-prompt-sam2/stage7/notebook_logs/1794426915340401511_bootstrap.log
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 68.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 25.8 MB/s  0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 17.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 64.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 94.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Revisar presupuesto cerrado

Ambos regímenes comparten espacio de búsqueda y controles antes de ver resultados nuevos. No cambiar esta matriz después de inspeccionar el régimen oficial. La diferencia de componentes y total real de puntos se reporta: igualdad de k por componente no garantiza igualdad de llamadas SAM.


In [ ]:
matrix = json.loads((REPO / 'configs/ablations.yaml').read_text('utf-8'))
print(json.dumps(matrix, ensure_ascii=False, indent=2))
print('13 filas x 89/75 chips val: 1157 + 975 = 2132 unidades lógicas, con cachés previas válidas reutilizadas.')
print('Perfil: primer chip val por régimen, 13 filas, 2 rutas cache, 5 warm-up + 20 repeticiones, batch una imagen.')
print('No hay entrenamiento ni nuevos candidatos fuera de esta matriz. GPU insuficiente = registrar pendiente.')


{
  "schema_version": 1,
  "stage": "phase1_stage7",
  "date": "2026-11-01",
  "regimes": [
    "official",
    "leave_paraguay_out"
  ],
  "roles": [
    "val"
  ],
  "seed": 42,
  "positive_sweep": [
    1,
    3,
    5,
    10
  ],
  "negative_sweep": [
    0,
    1,
    3,
    5
  ],
  "fixed_positive": 3,
  "fixed_negative": 3,
  "candidate_status": "R: k+ inherited from draft; k- new recommendation; closed common budget before execution",
  "calibration": {
    "execute": false,
    "source": "stage5_complete_bundle_and_stage6_best",
    "rule": "reuse_per_regime_probe_positive_threshold_grid_threshold_and_best_epoch_no_retuning"
  },
  "hypotheses": {
    "A1": {
      "date": "2026-11-01",
      "text": "H1 WAIMLAP: IoU puntos DINOv3 < referencia asistida e > malla. Caja completa es control espacial fijo. Puede refutarse.",
      "source": "docs/WAIMLAP/Abstract_CostOfThePrompt.md section 2"
    },
    "A2+": {
      "date": "2026-11-01",
      "text": "H2 WAIMLAP: incrementar 

## 3. Ejecutar prepare, resume o recover

Primera ejecución: `RUN_MODE='prepare'`. Si hay interrupción técnica, detener el escritor anterior y usar `resume` con el mismo paquete, raíz y condiciones; las generaciones completas se reutilizan y los fallos publicados no se ocultan.

Después de completar ambos regímenes, **guardar este notebook con salidas prepare**, desconectar/eliminar runtime y abrir otro runtime GPU. Repetir solo entorno y esta sección con `RUN_MODE='recover'`. Reconstruye offline desde imagen el primer chip val en cada fila exitosa, verifica todas las generaciones y recalcula selección/locks sin alterarlos. Guardar también ese notebook con salidas. No ejecutar prepare dos veces.


In [ ]:
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2/auto-prompt-sam2'
RUN_MODE = 'resume'
command([str(PYTHON), str(REPO / 'scripts/run_stage7.py'), RUN_MODE, '--root', PERSIST_ROOT], RUN_MODE + '.log')
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2/auto-prompt-sam2'
root = Path(PERSIST_ROOT)
assert (root / 'environment/runs/20261028T221055Z_1d0f6a92f4_aa07b10e').is_dir(), 'No se encuentra la carpeta compartida de Roberto.'
old_log = Path('/content/drive/MyDrive/auto-prompt-sam2/stage7/notebook_logs/1794416033876798824_prepare.log')
print('Últimas líneas de la sesión anterior:')
if old_log.is_file():
    print('\n'.join(old_log.read_text(encoding='utf-8', errors='replace').splitlines()[-28:]))
for regime in ('official', 'leave_paraguay_out'):
    for f in sorted((root / 'regimes' / regime / 'stage7').glob('*/prepared/*/COMPLETE.json')):
        print(regime, 'ejecución guardada:', f.parents[2].name)
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2/auto-prompt-sam2'
root = Path(PERSIST_ROOT)
old_log = Path('/content/drive/MyDrive/auto-prompt-sam2/stage7/notebook_logs/1794416033876798824_prepare.log')
print('Registro anterior:', old_log.is_file())
if old_log.is_file():
    print('\n'.join(old_log.read_text(encoding='utf-8', errors='replace').splitlines()[-28:]))
for regime in ('official', 'leave_paraguay_out'):
    print(regime, [f.parents[2].name for f in (root / 'regimes' / regime / 'stage7').glob('*/prepared/*/COMPLETE.json')])


Log persistente: /content/drive/MyDrive/auto-prompt-sam2/stage7/notebook_logs/1794428138344300453_resume.log
`torch_dtype` is deprecated! Use `dtype` instead!
official a1_points 1 / 89
official a1_points 21 / 89
official a1_points 41 / 89
official a1_points 61 / 89
official a1_points 81 / 89
official a1_points ok {'water_iou': 0.29861481371632737, 'precision': 0.32715268295226385, 'recall': 0.7739225645276981, 'dice': 0.45989743927494964, 'nonwater_iou': 0.7808819289476611, 'miou': 0.5397483713319942}
official a1_assisted 1 / 89
official a1_assisted 21 / 89
official a1_assisted 41 / 89
official a1_assisted 61 / 89
official a1_assisted 81 / 89
official a1_assisted ok {'water_iou': 0.15401975854556055, 'precision': 0.15502590446121636, 'recall': 0.9595652494519811, 'dice': 0.2669274202716866, 'nonwater_iou': 0.35013583341089183, 'miou': 0.2520777959782262}
official a1_whole_box 1 / 89
official a1_whole_box 21 / 89
official a1_whole_box 41 / 89
official a1_whole_box 61 / 89
official a1_wh

## 4. Ver resultados, exportar y detener

Se informa rendimiento por chip/evento, costo del prompt (también negativo), curvas A2, diferencias A3 y fallos. Los perfiles son del primer chip val, no estiman la latencia de toda la población; incluyen lectura SHA y las sincronizaciones de etapas, con cachés OS/Drive calientes. Se separa carga inicial y ejecución con/sin features; ambas recalculan embeddings SAM. SAM y DINO permanecen residentes, incluso cuando la rama no utiliza DINO.

Compartir notebook(s) con salidas, **dos ZIP terminados y SHA**, JSON selection/lock/validation/recovery/both_regimes y logs/failure. Los locks incluyen las 13 filas finales con hashes y `stage8_authorized=false` hasta revisión externa y cierre de ambos regímenes. No abrir reservados ni seguir con la demo.


In [ ]:
# Leer evidencia existente sin recrear ni sobrescribir el ZIP reparado.
root = Path(PERSIST_ROOT)
for regime in ('official', 'leave_paraguay_out'):
    for path in sorted((root / 'regimes' / regime / 'stage7').glob('*/evidence/selection.json')):
        selection = json.loads(path.read_text(encoding='utf-8'))
        archive = path.parent.parent / 'evidence.zip'
        print(regime, selection['status'], 'selección:', selection['selected_id'])
        for row in selection['rows']:
            if row['id'] in ('a1_points', 'a1_assisted', 'a3_tuned') and row['status'] == 'ok':
                print(row['id'], 'IoU:', row['metrics']['pooled']['values']['water_iou'])
        print('Evidencia:', archive, '| existe:', archive.is_file())

# Revisión independiente en CPU; solo etiquetas de validación.
import time
archives = [next((root / 'regimes' / r / 'stage7').glob('*/evidence.zip')) for r in ('official', 'leave_paraguay_out')]
review_path = LOG_DIR.parent / ('revision_stage7_' + str(time.time_ns()) + '.json')
command([str(PYTHON), str(REPO / 'scripts/review_stage7_evidence.py'), *map(str, archives), '--data-root', str(root), '--output', str(review_path)], 'review_stage7.log')
print('Revisión guardada:', review_path)


official computed_pending_external_review selección: a3_tuned
a1_points IoU: 0.29861481371632737
a1_assisted IoU: 0.15401975854556055
a3_tuned IoU: 0.4840668324093302
Evidencia: /content/drive/MyDrive/auto-prompt-sam2/auto-prompt-sam2/regimes/official/stage7/b80a42d71a7333be255b48e725149466e826bfbecf63f5241f4ea969661dad87/evidence.zip | existe: True
leave_paraguay_out computed_pending_external_review selección: a3_tuned
a1_points IoU: 0.31684167884819114
a1_assisted IoU: 0.15150601128267668
a3_tuned IoU: 0.5058154025637092
Evidencia: /content/drive/MyDrive/auto-prompt-sam2/auto-prompt-sam2/regimes/leave_paraguay_out/stage7/e2ca85a0ba166f5506d453d2016f7797a2ddacdeec4f4fbaaa24b146ae3012b3/evidence.zip | existe: True
Log persistente: /content/drive/MyDrive/auto-prompt-sam2/stage7/notebook_logs/1794435476494332854_review_stage7.log
